# 2단계 데이터 전처리 통합본

# A. 원본 데이터 통합

In [1]:
from pathlib import Path

CUR_DIR = Path.cwd().resolve()
BASE_DIR = CUR_DIR / "data"

QUARTERLY_DIR = BASE_DIR / "quarterly"
CLEAN_DIR = BASE_DIR / "quarterly_clean"
FEATURES_DIR = BASE_DIR / "features_d"
LABELS_DIR = BASE_DIR / "labels"
FINAL_DIR = BASE_DIR / "final"


REPORT_DIR = BASE_DIR
for directory in [
    BASE_DIR,
    QUARTERLY_DIR,
    CLEAN_DIR,
    FEATURES_DIR,
    LABELS_DIR,
    FINAL_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("BASE_DIR      :", BASE_DIR)
print("QUARTERLY_DIR :", QUARTERLY_DIR)
print("CLEAN_DIR     :", CLEAN_DIR)
print("FEATURES_DIR  :", FEATURES_DIR)
print("LABELS_DIR    :", LABELS_DIR)
print("FINAL_DIR     :", FINAL_DIR)

BASE_DIR      : .\data
QUARTERLY_DIR : .\data\quarterly
CLEAN_DIR     : .\data\quarterly_clean
FEATURES_DIR  : .\data\features_d
LABELS_DIR    : .\data\labels
FINAL_DIR     : .\data\final


In [2]:

import re
from pathlib import Path

import pandas as pd

RAW_DIR = BASE_DIR
OUT_DIR = BASE_DIR
QUARTERLY_DIR.mkdir(parents=True, exist_ok=True)

FILENAME_PATTERN = re.compile(r"소상공인시장진흥공단_상가\(상권\)정보_서울_(\d{4})(\d{2})\.csv")

MONTH_TO_QUARTER = {"03": "Q1", "06": "Q2", "09": "Q3", "12": "Q4"}

YEAR_RANGE = ("2022", "2025")
EXPECTED_FILE_COUNT = 16

ENCODING_CANDIDATES = ["cp949", "utf-8", "utf-8-sig", "euc-kr"]


def parse_year_quarter(filename: str) -> tuple[str, str]:
    m = FILENAME_PATTERN.match(filename)
    if not m:
        raise ValueError(f"파일명 패턴이 예상과 다릅니다: {filename}")
    year, month = m.group(1), m.group(2)
    if month not in MONTH_TO_QUARTER:
        raise ValueError(f"알 수 없는 월 코드 '{month}' (파일: {filename})")
    return year, MONTH_TO_QUARTER[month]


def debug_list_all_files_in_raw_dir() -> None:
    if not RAW_DIR.exists():
        print(f"[디버그] 경로 자체가 존재하지 않습니다: {RAW_DIR}")
        return

    all_files = sorted(RAW_DIR.iterdir())
    print(f"[디버그] {RAW_DIR} 안의 전체 항목 수: {len(all_files)}")
    for f in all_files[:30]:
        print(f"    {repr(f.name)}")
    if len(all_files) > 30:
        print(f"    ... 외 {len(all_files) - 30}개")


def list_raw_files() -> list[Path]:
    files = sorted(RAW_DIR.glob("소상공인시장진흥공단_상가(상권)정보_서울_*.csv"))

    if not files:
        print("[경고] 정확한 패턴으로 0개 발견. 폴더 내용을 그대로 출력합니다.")
        debug_list_all_files_in_raw_dir()

        loose_files = [
            f for f in RAW_DIR.iterdir()
            if f.is_file() and f.suffix.lower() in (".csv", ".cvs")
        ]
        if loose_files:
            print(f"[안내] 확장자(.csv/.cvs) 기준으로는 {len(loose_files)}개가 발견됩니다. "
                  f"파일명 패턴(FILENAME_PATTERN)을 실제 이름에 맞게 수정해야 할 수 있습니다.")
        return []

    print(f"[파일 탐색] 폴더 내 전체 {len(files)}개 발견")

    in_range, out_of_range = [], []
    for f in files:
        year, _ = parse_year_quarter(f.name)
        if YEAR_RANGE[0] <= year <= YEAR_RANGE[1]:
            in_range.append(f)
        else:
            out_of_range.append(f)

    if out_of_range:
        print(f"  [제외] 작업 범위({YEAR_RANGE[0]}~{YEAR_RANGE[1]}) 밖이라 제외된 파일 {len(out_of_range)}개:")
        for f in out_of_range:
            print(f"    - {f.name}")

    print(f"  [사용] 범위 내 파일 {len(in_range)}개")
    if len(in_range) != EXPECTED_FILE_COUNT:
        print(f"  [경고] {EXPECTED_FILE_COUNT}개를 예상했으나 {len(in_range)}개입니다. 목록을 확인하세요:")
        for f in in_range:
            print(f"    - {f.name}")

    return in_range


def read_csv_robust(path: Path) -> pd.DataFrame:
    last_err = None
    for enc in ENCODING_CANDIDATES:
        try:
            df = pd.read_csv(path, encoding=enc, low_memory=False)
            print(f"    인코딩 '{enc}' 로 읽기 성공 (shape={df.shape})")
            return df
        except (UnicodeDecodeError, UnicodeError) as e:
            last_err = e
            continue
    raise RuntimeError(f"{path.name} 인코딩 판별 실패: {last_err}")


def process_one_quarter(path: Path) -> tuple[str, pd.DataFrame]:
    year, quarter = parse_year_quarter(path.name)
    key = f"{year}{quarter}"
    print(f"[처리] {path.name} -> {key}")

    df = read_csv_robust(path)
    df["기준_년"] = year
    df["기준_분기"] = quarter
    df["기준_년분기"] = key

    return key, df


def save_dataframe(df: pd.DataFrame, path_stem: Path) -> Path:
    parquet_path = path_stem.with_suffix(".parquet")
    try:
        df.to_parquet(parquet_path, index=False)
        return parquet_path
    except ImportError as e:
        print(f"    [경고] parquet 저장 실패({e}) -> pickle로 대체 저장합니다.")
        pickle_path = path_stem.with_suffix(".pkl")
        df.to_pickle(pickle_path)
        return pickle_path


def check_schema_consistency(columns_by_quarter: dict[str, set]) -> None:
    if not columns_by_quarter:
        print("[스킵] 비교할 분기 데이터가 없습니다 (파일을 하나도 못 읽었음).")
        return

    log_lines = []
    base_key = next(iter(columns_by_quarter))
    base_cols = columns_by_quarter[base_key]

    for key, cols in columns_by_quarter.items():
        if cols != base_cols:
            missing = base_cols - cols
            extra = cols - base_cols
            line = f"[스키마 불일치] {key}: 기준({base_key}) 대비 없는 컬럼={missing}, 추가 컬럼={extra}"
            print(line)
            log_lines.append(line)


def run_step_a() -> None:
    files = list_raw_files()

    if not files:
        print("\n[중단] 처리할 파일이 없어 종료합니다. 위 디버그 출력을 보고 "
              "RAW_DIR 경로 또는 FILENAME_PATTERN을 실제 파일명에 맞게 수정한 뒤 다시 실행하세요.")
        return

    columns_by_quarter: dict[str, set] = {}
    row_counts: dict[str, int] = {}

    for path in files:
        key, df = process_one_quarter(path)

        columns_by_quarter[key] = set(df.columns)
        row_counts[key] = len(df)

        out_path = save_dataframe(df, QUARTERLY_DIR / key)
        print(f"    저장 완료: {out_path} (행 수: {len(df):,})")

        del df  

    print("\n=== 분기별 행 수 요약 ===")
    for key, cnt in row_counts.items():
        print(f"  {key}: {cnt:,}행")
    print(f"  전체 합계: {sum(row_counts.values()):,}행 ")

    print("\n=== 스키마 일관성 점검 ===")
    check_schema_consistency(columns_by_quarter)


if __name__ == "__main__":
    run_step_a()

[파일 탐색] 폴더 내 전체 20개 발견
  [제외] 작업 범위(2022~2025) 밖이라 제외된 파일 4개:
    - 소상공인시장진흥공단_상가(상권)정보_서울_202103.csv
    - 소상공인시장진흥공단_상가(상권)정보_서울_202106.csv
    - 소상공인시장진흥공단_상가(상권)정보_서울_202109.csv
    - 소상공인시장진흥공단_상가(상권)정보_서울_202112.csv
  [사용] 범위 내 파일 16개
[처리] 소상공인시장진흥공단_상가(상권)정보_서울_202203.csv -> 2022Q1
    인코딩 'utf-8' 로 읽기 성공 (shape=(524728, 39))
    저장 완료: .\data\quarterly\2022Q1.parquet (행 수: 524,728)
[처리] 소상공인시장진흥공단_상가(상권)정보_서울_202206.csv -> 2022Q2
    인코딩 'utf-8' 로 읽기 성공 (shape=(529441, 39))
    저장 완료: .\data\quarterly\2022Q2.parquet (행 수: 529,441)
[처리] 소상공인시장진흥공단_상가(상권)정보_서울_202209.csv -> 2022Q3
    인코딩 'utf-8' 로 읽기 성공 (shape=(534473, 39))
    저장 완료: .\data\quarterly\2022Q3.parquet (행 수: 534,473)
[처리] 소상공인시장진흥공단_상가(상권)정보_서울_202212.csv -> 2022Q4
    인코딩 'utf-8' 로 읽기 성공 (shape=(534294, 39))
    저장 완료: .\data\quarterly\2022Q4.parquet (행 수: 534,294)
[처리] 소상공인시장진흥공단_상가(상권)정보_서울_202303.csv -> 2023Q1
    인코딩 'utf-8' 로 읽기 성공 (shape=(474260, 39))
    저장 완료: .\data\quarterly\2023Q1.parquet (행 수: 474,260)

# B. 폐업 라벨 생성

In [3]:


from pathlib import Path

import pandas as pd



def log(msg: str = "") -> None:
    print(msg)


ID_COL = "상가업소번호"

NEEDED_COLS = [ID_COL, "상호명", "상권업종소분류코드", "건물관리번호", "경도", "위도"]

def _resolve_quarter_path(key: str) -> Path:
    pkl_path = QUARTERLY_DIR / f"{key}.pkl"
    if pkl_path.exists():
        return pkl_path
    parquet_path = QUARTERLY_DIR / f"{key}.parquet"
    if parquet_path.exists():
        return parquet_path
    raise FileNotFoundError(f"{key}에 해당하는 quarterly 파일을 찾을 수 없습니다.")


def load_quarter_reduced(key: str) -> pd.DataFrame:
    path = _resolve_quarter_path(key)
    if path.suffix == ".pkl":
        df = pd.read_pickle(path)
    else:
        df = pd.read_parquet(path)

    df = df[NEEDED_COLS].copy()

    dup_count = df[ID_COL].duplicated().sum()
    if dup_count > 0:
        log(f"  [경고] {key}: 상가업소번호 중복 {dup_count}건 발견 -> 첫 행만 유지")
        df = df.drop_duplicates(subset=[ID_COL], keep="first")

    df = df.set_index(ID_COL)
    return df


def load_all_quarters(keys: list[str]) -> dict[str, pd.DataFrame]:
    quarters = {}
    prev_count = None
    for key in keys:
        log(f"[로드] {key}")
        quarters[key] = load_quarter_reduced(key)
        cnt = len(quarters[key])
        log(f"    -> {cnt:,}행 (축소 컬럼 {len(NEEDED_COLS) - 1}개)")

        if prev_count is not None and prev_count > 0:
            change_ratio = (cnt - prev_count) / prev_count
            if abs(change_ratio) >= 0.05:
                log(f"    [주의] 직전 분기 대비 {change_ratio:+.1%} 변동 -> 정상적인 개업/폐업 규모를 "
                    f"넘어설 수 있으니 원본 데이터 확인 필요")
        prev_count = cnt
    return quarters


def find_relocations(disappeared_ids: set, df_t: pd.DataFrame, df_next: pd.DataFrame) -> set:
    appeared_ids = set(df_next.index) - set(df_t.index)
    if not disappeared_ids or not appeared_ids:
        return set()

    match_cols = ["상호명", "상권업종소분류코드", "건물관리번호"]

    disappeared_keys = df_t.loc[list(disappeared_ids), match_cols].dropna(subset=["건물관리번호"])
    appeared_keys = df_next.loc[list(appeared_ids), match_cols].dropna(subset=["건물관리번호"])

    merged = disappeared_keys.reset_index().merge(
        appeared_keys.reset_index(),
        on=match_cols,
        suffixes=("_old", "_new"),
    )
    relocated_ids = set(merged[f"{ID_COL}_old"])
    return relocated_ids


def find_reappeared(remaining_ids: set, key_t2: str | None, quarters: dict[str, pd.DataFrame]) -> set:
    if key_t2 is None or key_t2 not in quarters:
        return set()
    ids_t2 = set(quarters[key_t2].index)
    return remaining_ids & ids_t2


def detect_outlier_quarters(quarter_stats: list[dict]) -> set:
    rates = pd.Series([s["disappear_rate"] for s in quarter_stats])
    q1, q3 = rates.quantile(0.25), rates.quantile(0.75)
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr

    outlier_keys = set()
    for s in quarter_stats:
        if s["disappear_rate"] > upper_bound:
            outlier_keys.add(s["key_t"])
    log(f"\n[이상치 분기 탐지] 1차 폐업후보 비율 상한선(IQR*1.5) = {upper_bound:.2%}")
    for s in quarter_stats:
        marker = " <- 이상치" if s["key_t"] in outlier_keys else ""
        log(f"    {s['key_t']}: 폐업후보 비율 {s['disappear_rate']:.2%}{marker}")
    return outlier_keys


def generate_labels_for_quarter(
    key_t: str,
    key_next: str,
    key_t2: str | None,
    quarters: dict[str, pd.DataFrame],
) -> tuple[pd.DataFrame, dict]:
    df_t = quarters[key_t]
    df_next = quarters[key_next]

    ids_t = set(df_t.index)
    ids_next = set(df_next.index)

    disappeared = ids_t - ids_next
    stayed = ids_t & ids_next

    log(f"[{key_t} -> {key_next}] 전체 {len(ids_t):,} / 유지 {len(stayed):,} / 1차 폐업후보 {len(disappeared):,}")

    relocated = find_relocations(disappeared, df_t, df_next)
    remaining_after_relocation = disappeared - relocated
    log(f"    이전(주소이동) 필터로 제외: {len(relocated):,} -> 잔여 후보 {len(remaining_after_relocation):,}")

    reappeared = find_reappeared(remaining_after_relocation, key_t2, quarters)
    remaining_after_gap = remaining_after_relocation - reappeared
    log(f"    데이터갭 보정으로 제외: {len(reappeared):,} -> 최종 폐업 확정 {len(remaining_after_gap):,}")

    records = []
    for sid in ids_t:
        if sid in remaining_after_gap:
            label, note = 1, "폐업"
        elif sid in relocated:
            label, note = 0, "이전추정"
        elif sid in reappeared:
            label, note = 0, "데이터갭보정"
        else:
            label, note = 0, "생존"
        records.append((sid, key_t, label, note))

    result = pd.DataFrame(records, columns=[ID_COL, "기준_년분기", "폐업_다음분기", "라벨_비고"])

    stats = {
        "key_t": key_t,
        "total": len(ids_t),
        "disappear_rate": len(disappeared) / len(ids_t) if len(ids_t) else 0.0,
    }
    return result, stats


def run_step_b() -> None:
    keys = sorted(
        p.stem for p in QUARTERLY_DIR.glob("*")
        if p.suffix in (".pkl", ".parquet")
    )
    log(f"[분기 목록] {keys}")

    quarters = load_all_quarters(keys)

    all_labels = []
    quarter_stats = []
    for i in range(len(keys) - 1):
        key_t = keys[i]
        key_next = keys[i + 1]
        key_t2 = keys[i + 2] if i + 2 < len(keys) else None

        labels, stats = generate_labels_for_quarter(key_t, key_next, key_t2, quarters)
        quarter_stats.append(stats)

        out_path = LABELS_DIR / f"{key_t}_labels.pkl"
        labels.to_pickle(out_path)
        log(f"    저장 완료: {out_path} ({len(labels):,}행)\n")
        all_labels.append(labels)

    last_key = keys[-1]
    log(f"[안내] {last_key}는 다음 분기 데이터가 없어 라벨 생성 대상에서 제외됨 (예측 시점으로만 사용)")

    outlier_keys = detect_outlier_quarters(quarter_stats)

    combined = pd.concat(all_labels, ignore_index=True)
    combined["이상치_의심분기"] = combined["기준_년분기"].isin(outlier_keys)

    combined_path = LABELS_DIR / "all_labels_combined.pkl"
    combined.to_pickle(combined_path)
    log(f"\n[전체 통합 저장] {combined_path} (총 {len(combined):,}행)")

    log("\n=== 라벨 분포 요약 (전체) ===")
    log(combined["라벨_비고"].value_counts())
    log(f"\n최종 폐업(1) 비율 (전체): {combined['폐업_다음분기'].mean():.4%}")

    log("\n=== 라벨 분포 요약 (이상치 분기 제외) ===")
    clean = combined[~combined["이상치_의심분기"]]
    log(clean["라벨_비고"].value_counts())
    log(f"\n최종 폐업(1) 비율 (이상치 제외): {clean['폐업_다음분기'].mean():.4%}")



if __name__ == "__main__":
    run_step_b()

[분기 목록] ['2022Q1', '2022Q2', '2022Q3', '2022Q4', '2023Q1', '2023Q2', '2023Q3', '2023Q4', '2024Q1', '2024Q2', '2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4']
[로드] 2022Q1
    -> 524,728행 (축소 컬럼 5개)
[로드] 2022Q2
    -> 529,441행 (축소 컬럼 5개)
[로드] 2022Q3
    -> 534,473행 (축소 컬럼 5개)
[로드] 2022Q4
    -> 534,294행 (축소 컬럼 5개)
[로드] 2023Q1
    -> 474,260행 (축소 컬럼 5개)
    [주의] 직전 분기 대비 -11.2% 변동 -> 정상적인 개업/폐업 규모를 넘어설 수 있으니 원본 데이터 확인 필요
[로드] 2023Q2
    -> 489,903행 (축소 컬럼 5개)
[로드] 2023Q3
    -> 482,959행 (축소 컬럼 5개)
[로드] 2023Q4
    -> 489,872행 (축소 컬럼 5개)
[로드] 2024Q1
    -> 484,226행 (축소 컬럼 5개)
[로드] 2024Q2
    -> 477,378행 (축소 컬럼 5개)
[로드] 2024Q3
    -> 466,022행 (축소 컬럼 5개)
[로드] 2024Q4
    -> 553,535행 (축소 컬럼 5개)
    [주의] 직전 분기 대비 +18.8% 변동 -> 정상적인 개업/폐업 규모를 넘어설 수 있으니 원본 데이터 확인 필요
[로드] 2025Q1
    -> 540,516행 (축소 컬럼 5개)
[로드] 2025Q2
    -> 536,114행 (축소 컬럼 5개)
[로드] 2025Q3
    -> 536,261행 (축소 컬럼 5개)
[로드] 2025Q4
    -> 534,978행 (축소 컬럼 5개)
[2022Q1 -> 2022Q2] 전체 524,728 / 유지 511,210 / 1차 폐업후보 13,518
    이전(주소

In [4]:
from pathlib import Path

import pandas as pd

ID_COL = "상가업소번호"
NEEDED_COLS = [ID_COL, "상호명", "상권업종소분류코드", "건물관리번호", "경도", "위도", "도로명주소"]


def load_quarter(key: str) -> pd.DataFrame:
    pkl_path = QUARTERLY_DIR / f"{key}.pkl"
    parquet_path = QUARTERLY_DIR / f"{key}.parquet"

    if pkl_path.exists():
        df = pd.read_pickle(pkl_path)
    elif parquet_path.exists():
        df = pd.read_parquet(parquet_path)
    else:
        raise FileNotFoundError(
            f"{key} 파일을 찾을 수 없습니다.\n"
            f"- {pkl_path}\n"
            f"- {parquet_path}"
        )

    return df[NEEDED_COLS]


def diagnose(key_t: str, key_next: str, n_samples: int = 20, seed: int = 42) -> None:
    df_t = load_quarter(key_t)
    df_next = load_quarter(key_next)

    labels = pd.read_pickle(LABELS_DIR / f"{key_t}_labels.pkl")
    closed_ids = labels.loc[labels["라벨_비고"] == "폐업", ID_COL]

    sample_ids = closed_ids.sample(n=min(n_samples, len(closed_ids)), random_state=seed)

    df_t_idx = df_t.set_index(ID_COL)
    df_next_by_building = df_next.set_index("건물관리번호")

    print(f"=== {key_t} -> {key_next} '폐업 확정' 샘플 {len(sample_ids)}건 진단 ===\n")

    for sid in sample_ids:
        row = df_t_idx.loc[sid]
        building = row["건물관리번호"]
        print(f"[{sid}] {key_t} 정보: 상호명={row['상호명']!r}, 업종={row['상권업종소분류코드']}, "
              f"주소={row['도로명주소']}")

        if building in df_next_by_building.index:
            same_building_next = df_next_by_building.loc[[building]]
            print(f"    -> {key_next} 같은 건물({building})에 {len(same_building_next)}개 업소 존재:")
            for _, r in same_building_next.iterrows():
                print(f"       상호명={r['상호명']!r}, 업종={r['상권업종소분류코드']}")
        else:
            print(f"    -> {key_next}에 같은 건물({building}) 자체가 없음 (건물 통째로 사라짐 또는 건물관리번호 결측)")
        print()


if __name__ == "__main__":
    print("################ 2022Q4 -> 2023Q1 진단 ################")
    diagnose("2022Q4", "2023Q1", n_samples=20)

    print("\n\n################ 2024Q3 -> 2024Q4 진단 ################")
    diagnose("2024Q3", "2024Q4", n_samples=20)

################ 2022Q4 -> 2023Q1 진단 ################
=== 2022Q4 -> 2023Q1 '폐업 확정' 샘플 20건 진단 ===

[MA0106202201A0521441] 2022Q4 정보: 상호명='신세계인터내셔날', 업종=G20905, 주소=서울특별시 강남구 압구정로80길 26
    -> 2023Q1 같은 건물(1168010400100950004027870)에 6개 업소 존재:
       상호명='씨유청담노아점', 업종=G20405
       상호명='아이프앤코', 업종=G21801
       상호명='제이앤영글로벌', 업종=G21801
       상호명='주영갤러리', 업종=G21801
       상호명='비금도', 업종=I20101
       상호명='호리팩토리', 업종=G21801

[MA0101202410A0149513] 2022Q4 정보: 상호명='레마뮤직，레마엔터테인먼트', 업종=P10501, 주소=서울특별시 영등포구 문래로10길 20-1
    -> 2023Q1에 같은 건물(1156012400100220003022360) 자체가 없음 (건물 통째로 사라짐 또는 건물관리번호 결측)

[MA0101202410A0262510] 2022Q4 정보: 상호명='소리엘음악교습소', 업종=P10609, 주소=서울특별시 중랑구 동일로126길 42
    -> 2023Q1에 같은 건물(1126010300102950001000001) 자체가 없음 (건물 통째로 사라짐 또는 건물관리번호 결측)

[MA0101202410A0672687] 2022Q4 정보: 상호명='프랜시스파커서초어학원', 업종=P10615, 주소=서울특별시 서초구 효령로 49
    -> 2023Q1에 같은 건물(1165010100109800029012872) 자체가 없음 (건물 통째로 사라짐 또는 건물관리번호 결측)

[MA0102202408B0057308] 2022Q4 정보: 상호명='매쓰피아학원', 업종=P10501, 주소=서울특별시 강

# C. 전처리

In [7]:
from pathlib import Path

import pandas as pd




def log(msg: str = "") -> None:
    print(msg)




def _resolve_quarter_path(key: str) -> Path:
    pkl_path = QUARTERLY_DIR / f"{key}.pkl"
    if pkl_path.exists():
        return pkl_path
    parquet_path = QUARTERLY_DIR / f"{key}.parquet"
    if parquet_path.exists():
        return parquet_path
    raise FileNotFoundError(f"{key}에 해당하는 quarterly 파일을 찾을 수 없습니다.")


def load_quarter(key: str) -> pd.DataFrame:
    path = _resolve_quarter_path(key)
    if path.suffix == ".pkl":
        return pd.read_pickle(path)
    return pd.read_parquet(path)


def compute_missing_summary(key: str) -> pd.Series:
    df = load_quarter(key)
    summary = df.isna().mean()
    summary.name = key
    row_count = len(df)
    del df
    return summary, row_count


def run_missing_diagnosis() -> None:
    keys = sorted(
        p.stem for p in QUARTERLY_DIR.glob("*")
        if p.suffix in (".pkl", ".parquet")
    )
    log(f"[분기 목록] {keys}\n")

    summaries = []
    row_counts = {}
    for key in keys:
        log(f"[처리] {key}")
        summary, row_count = compute_missing_summary(key)
        summaries.append(summary)
        row_counts[key] = row_count

    table = pd.concat(summaries, axis=1).T
    table.index.name = "기준_년분기"

    csv_path = OUT_DIR / "missing_rate_by_quarter.csv"
    table.to_csv(csv_path, encoding="utf-8-sig")
    log(f"\n[저장 완료] {csv_path}")

    col_avg = table.mean().sort_values(ascending=False)
    col_range = (table.max() - table.min()).reindex(col_avg.index)

    log("\n=== 컬럼별 평균 결측률 (내림차순) 및 분기 간 변동폭 ===")
    log(f"{'컬럼명':<20}{'평균 결측률':>12}{'변동폭(최대-최소)':>20}")
    for col in col_avg.index:
        avg = col_avg[col]
        rng = col_range[col]
        flag = ""
        if avg > 0.20:
            flag += " [평균 20%+ 결측]"
        if rng > 0.10:
            flag += " [분기별 변동 10%p+ -> 시기별 수집방식 차이 의심]"
        log(f"{col:<20}{avg:>11.2%}{rng:>19.2%}{flag}")

    log("\n=== 분기별 전체 행 수 (참고) ===")
    for key, cnt in row_counts.items():
        log(f"  {key}: {cnt:,}행")



if __name__ == "__main__":
    run_missing_diagnosis()

[분기 목록] ['2022Q1', '2022Q2', '2022Q3', '2022Q4', '2023Q1', '2023Q2', '2023Q3', '2023Q4', '2024Q1', '2024Q2', '2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4']

[처리] 2022Q1
[처리] 2022Q2
[처리] 2022Q3
[처리] 2022Q4
[처리] 2023Q1
[처리] 2023Q2
[처리] 2023Q3
[처리] 2023Q4
[처리] 2024Q1
[처리] 2024Q2
[처리] 2024Q3
[처리] 2024Q4
[처리] 2025Q1
[처리] 2025Q2
[처리] 2025Q3
[처리] 2025Q4

[저장 완료] .\data\missing_rate_by_quarter.csv

=== 컬럼별 평균 결측률 (내림차순) 및 분기 간 변동폭 ===
컬럼명                       평균 결측률          변동폭(최대-최소)
호정보                     100.00%              0.00% [평균 20%+ 결측]
동정보                      97.04%              9.94% [평균 20%+ 결측]
지점명                      94.36%              8.72% [평균 20%+ 결측]
건물부번지                    87.01%              0.60% [평균 20%+ 결측]
건물명                      52.18%              2.15% [평균 20%+ 결측]
층정보                      32.17%              4.59% [평균 20%+ 결측]
지번부번지                    16.97%              1.32%
건물관리번호                    0.23%              0.66%
표준산업분류명          

In [8]:
import re
from pathlib import Path

import numpy as np
import pandas as pd




def log(msg: str = "") -> None:
    print(msg)



def _resolve_quarter_path(key: str) -> Path:
    pkl_path = QUARTERLY_DIR / f"{key}.pkl"
    if pkl_path.exists():
        return pkl_path
    parquet_path = QUARTERLY_DIR / f"{key}.parquet"
    if parquet_path.exists():
        return parquet_path
    raise FileNotFoundError(f"{key}에 해당하는 quarterly 파일을 찾을 수 없습니다.")


def load_quarter(key: str) -> pd.DataFrame:
    path = _resolve_quarter_path(key)
    if path.suffix == ".pkl":
        return pd.read_pickle(path)
    return pd.read_parquet(path)



_BASEMENT_PATTERN = re.compile(r"^(지하|B|b)(\d+)층?$")
_GROUND_EXPLICIT_PATTERN = re.compile(r"^지상(\d+)층?$")
_NEGATIVE_NUMBER_PATTERN = re.compile(r"^-(\d+)$")
_FLOOR_SUFFIX_PATTERN = re.compile(r"^(\d+)층$")
_PURE_NUMBER = re.compile(r"^\d+$")

_BASEMENT_UNKNOWN_DEPTH_TEXTS = {"지", "지층"}
_SEMI_BASEMENT_TEXTS = {"반", "반지층", "반지하"}

PLAUSIBLE_FLOOR_MIN = -15
PLAUSIBLE_FLOOR_MAX = 130


def parse_floor(raw) -> dict:
    result = {
        "층수_숫자": np.nan,
        "지하여부": 0,
        "반지하여부": 0,
        "지하층수미상여부": 0,
        "층호결합추정여부": 0,
        "층정보파싱실패여부": False,
    }

    text = str(raw).strip().replace(" ", "")

    if text in _BASEMENT_UNKNOWN_DEPTH_TEXTS:
        result["지하여부"] = 1
        result["지하층수미상여부"] = 1
        return result

    if text in _SEMI_BASEMENT_TEXTS:
        result["반지하여부"] = 1
        return result

    sign = 1
    body = None

    m = _BASEMENT_PATTERN.match(text)
    if m:
        sign = -1
        body = m.group(2)
    else:
        m = _GROUND_EXPLICIT_PATTERN.match(text)
        if m:
            body = m.group(1)
        else:
            m = _NEGATIVE_NUMBER_PATTERN.match(text)
            if m:
                sign = -1
                body = m.group(1)
            else:
                m = _FLOOR_SUFFIX_PATTERN.match(text)
                if m:
                    body = m.group(1)
                elif _PURE_NUMBER.match(text):
                    body = text

    if body is None:
        result["층정보파싱실패여부"] = True
        return result

    n_digits = len(body)
    value = int(body)

    if n_digits <= 2:
        floor = value
    elif n_digits in (3, 4):
        floor = value // 100
        result["층호결합추정여부"] = 1
    else:
        result["층정보파싱실패여부"] = True
        return result

    floor = floor * sign

    if not (PLAUSIBLE_FLOOR_MIN <= floor <= PLAUSIBLE_FLOOR_MAX):
        result["층정보파싱실패여부"] = True
        return result

    result["층수_숫자"] = float(floor)
    result["지하여부"] = 1 if floor < 0 else 0
    return result


def build_floor_features(series: pd.Series) -> pd.DataFrame:
    missing_mask = series.isna()

    records = []
    for raw in series:
        if pd.isna(raw):
            records.append({
                "층수_숫자": np.nan, "지하여부": 0, "반지하여부": 0,
                "지하층수미상여부": 0, "층호결합추정여부": 0, "층정보파싱실패여부": False,
            })
            continue
        records.append(parse_floor(raw))

    parsed = pd.DataFrame(records, index=series.index)

    return pd.DataFrame({
        "층수_숫자": parsed["층수_숫자"],
        "지상1층여부": (parsed["층수_숫자"] == 1).astype(int),
        "지하여부": parsed["지하여부"].astype(int),
        "반지하여부": parsed["반지하여부"].astype(int),
        "지하층수미상여부": parsed["지하층수미상여부"].astype(int),
        "층호결합추정여부": parsed["층호결합추정여부"].astype(int),
        "층정보파싱실패여부": parsed["층정보파싱실패여부"].astype(int),
        "층정보결측여부": missing_mask.astype(int),
    })

DROP_COLS = ["호정보", "동정보", "건물부번지", "지번부번지"]


def clean_one_quarter(key: str) -> pd.DataFrame:
    df = load_quarter(key)
    n_before = len(df)
    cols_before = set(df.columns)

    existing_drop_cols = [c for c in DROP_COLS if c in df.columns]
    df = df.drop(columns=existing_drop_cols)

    df["지점여부"] = df["지점명"].notna().astype(int)
    df = df.drop(columns=["지점명"])

    df["건물명존재여부"] = df["건물명"].notna().astype(int)
    df = df.drop(columns=["건물명"])

    floor_features = build_floor_features(df["층정보"])
    df = pd.concat([df.drop(columns=["층정보"]), floor_features], axis=1)

    n_after = len(df)
    cols_after = set(df.columns)

    log(f"[{key}] 행 수 {n_before:,} -> {n_after:,} (변화 없어야 정상)")
    log(f"    드롭된 컬럼: {sorted(cols_before - cols_after)}")
    log(f"    추가된 컬럼: {sorted(cols_after - cols_before)}")
    log(f"    지점여부=1 비율: {df['지점여부'].mean():.2%}")
    log(f"    건물명존재여부=1 비율: {df['건물명존재여부'].mean():.2%}")
    log(f"    층정보결측여부=1 비율: {df['층정보결측여부'].mean():.2%}")
    log(f"    층정보파싱실패여부=1 비율: {df['층정보파싱실패여부'].mean():.2%}")
    log(f"    층호결합추정여부=1 비율: {df['층호결합추정여부'].mean():.2%} (예: '202'->2층으로 추정 변환된 비율)")
    log(f"    지하여부=1 비율: {df['지하여부'].mean():.2%}")
    log(f"    지하층수미상여부=1 비율: {df['지하층수미상여부'].mean():.2%} ('지'/'지층'처럼 정확한 깊이 모르는 경우)")
    log(f"    반지하여부=1 비율: {df['반지하여부'].mean():.2%}")
    log(f"    지상1층여부=1 비율: {df['지상1층여부'].mean():.2%}")
    log(f"    층수_숫자 범위: min={df['층수_숫자'].min()}, max={df['층수_숫자'].max()} "
        f"(±{PLAUSIBLE_FLOOR_MAX}층 밖은 파싱실패로 처리되어 여기 안 나와야 정상)")

    assert n_before == n_after, f"{key}: 전처리 중 행 수가 변했습니다 (버그 의심)"

    return df


def run_step_c() -> None:
    keys = sorted(
        p.stem for p in QUARTERLY_DIR.glob("*")
        if p.suffix in (".pkl", ".parquet")
    )
    log(f"[분기 목록] {keys}\n")

    for key in keys:
        cleaned = clean_one_quarter(key)
        out_path = CLEAN_DIR / f"{key}.pkl"
        cleaned.to_pickle(out_path)
        log(f"    저장 완료: {out_path}\n")
        del cleaned



if __name__ == "__main__":
    run_step_c()

[분기 목록] ['2022Q1', '2022Q2', '2022Q3', '2022Q4', '2023Q1', '2023Q2', '2023Q3', '2023Q4', '2024Q1', '2024Q2', '2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4']

[2022Q1] 행 수 524,728 -> 524,728 (변화 없어야 정상)
    드롭된 컬럼: ['건물명', '건물부번지', '동정보', '지번부번지', '지점명', '층정보', '호정보']
    추가된 컬럼: ['건물명존재여부', '반지하여부', '지상1층여부', '지점여부', '지하여부', '지하층수미상여부', '층수_숫자', '층정보결측여부', '층정보파싱실패여부', '층호결합추정여부']
    지점여부=1 비율: 8.31%
    건물명존재여부=1 비율: 47.06%
    층정보결측여부=1 비율: 34.43%
    층정보파싱실패여부=1 비율: 0.00%
    층호결합추정여부=1 비율: 0.05% (예: '202'->2층으로 추정 변환된 비율)
    지하여부=1 비율: 2.02%
    지하층수미상여부=1 비율: 1.60% ('지'/'지층'처럼 정확한 깊이 모르는 경우)
    반지하여부=1 비율: 0.00%
    지상1층여부=1 비율: 34.64%
    층수_숫자 범위: min=-7.0, max=86.0 (±130층 밖은 파싱실패로 처리되어 여기 안 나와야 정상)
    저장 완료: .\data\quarterly_clean\2022Q1.pkl

[2022Q2] 행 수 529,441 -> 529,441 (변화 없어야 정상)
    드롭된 컬럼: ['건물명', '건물부번지', '동정보', '지번부번지', '지점명', '층정보', '호정보']
    추가된 컬럼: ['건물명존재여부', '반지하여부', '지상1층여부', '지점여부', '지하여부', '지하층수미상여부', '층수_숫자', '층정보결측여부', '층정보파싱실패여부', '층호결합추정여부

In [9]:
from pathlib import Path

import pandas as pd
        



def log(msg=""):
    print(msg)



SUSPICIOUS_GROUND_FLOOR_ABOVE = 70   
SUSPICIOUS_BASEMENT_BELOW = -10      


def resolve_path(base_dir: Path, key: str) -> Path:
    for suffix in (".pkl", ".parquet"):
        p = base_dir / f"{key}{suffix}"
        if p.exists():
            return p
    raise FileNotFoundError(f"{key} 파일을 {base_dir}에서 찾을 수 없습니다.")


def load(base_dir: Path, key: str) -> pd.DataFrame:
    path = resolve_path(base_dir, key)
    if path.suffix == ".pkl":
        return pd.read_pickle(path)
    return pd.read_parquet(path)


def check_distribution_all_quarters() -> None:
    keys = sorted(p.stem for p in CLEAN_DIR.glob("*.pkl"))
    rows = []

    for key in keys:
        df = load(CLEAN_DIR, key)
        floor = df["층수_숫자"].dropna()

        n_total = len(df)
        n_valid = len(floor)
        n_suspicious_high = (floor > SUSPICIOUS_GROUND_FLOOR_ABOVE).sum()
        n_suspicious_low = (floor < SUSPICIOUS_BASEMENT_BELOW).sum()

        rows.append({
            "기준_년분기": key,
            "min": floor.min(),
            "max": floor.max(),
            "p50": floor.median(),
            "p99": floor.quantile(0.99),
            f"지상{SUSPICIOUS_GROUND_FLOOR_ABOVE}층초과_건수": n_suspicious_high,
            f"지하{-SUSPICIOUS_BASEMENT_BELOW}층초과_건수": n_suspicious_low,
            "유효값_비율": n_valid / n_total,
        })
        del df

    summary = pd.DataFrame(rows)
    log("=== 분기별 층수_숫자 분포 요약 ===")
    log(summary.to_string(index=False))

    out_path = OUT_DIR / "floor_distribution_summary.csv"
    summary.to_csv(out_path, index=False, encoding="utf-8-sig")
    log(f"\n[저장 완료] {out_path}")

    log("\n=== 의심 구간 실제 값 샘플 (전체 분기 통틀어) ===")
    all_suspicious = []
    for key in keys:
        df = load(CLEAN_DIR, key)
        sus = df[(df["층수_숫자"] > SUSPICIOUS_GROUND_FLOOR_ABOVE) |
                 (df["층수_숫자"] < SUSPICIOUS_BASEMENT_BELOW)]
        if len(sus) > 0:
            all_suspicious.append(sus[["상가업소번호", "층수_숫자"]].assign(기준_년분기=key))
        del df

    if all_suspicious:
        combined = pd.concat(all_suspicious, ignore_index=True)
        log(combined.head(30).to_string(index=False))
        log(f"\n(전체 의심 건수: {len(combined)}건)")
    else:
        log("의심 구간에 해당하는 값 없음.")


def sample_parse_failures(key: str, n_samples: int = 30) -> None:
    raw = load(QUARTERLY_DIR, key)[["상가업소번호", "층정보"]]
    clean = load(CLEAN_DIR, key)[["상가업소번호", "층정보파싱실패여부"]]

    merged = raw.merge(clean, on="상가업소번호", how="inner")
    failed = merged[merged["층정보파싱실패여부"] == 1]

    log(f"\n=== {key} 파싱실패 원본 텍스트 빈도 (상위 {n_samples}개 고유값) ===")
    log(failed["층정보"].value_counts().head(n_samples).to_string())


def inspect_raw_text_for_suspicious(key: str, n_samples: int = 20) -> None:
    raw = load(QUARTERLY_DIR, key)[["상가업소번호", "층정보", "도로명주소"]]
    clean = load(CLEAN_DIR, key)[["상가업소번호", "층수_숫자"]]

    merged = raw.merge(clean, on="상가업소번호", how="inner")
    suspicious = merged[(merged["층수_숫자"] > SUSPICIOUS_GROUND_FLOOR_ABOVE) |
                         (merged["층수_숫자"] < SUSPICIOUS_BASEMENT_BELOW)]

    log(f"\n=== {key} 의심 층수값의 원본 텍스트 ({len(suspicious)}건 중 {n_samples}건 샘플) ===")
    log(suspicious[["상가업소번호", "층정보", "층수_숫자", "도로명주소"]].head(n_samples).to_string(index=False))


if __name__ == "__main__":
    check_distribution_all_quarters()
    inspect_raw_text_for_suspicious("2022Q1", n_samples=20)
    sample_parse_failures("2022Q1", n_samples=30)


=== 분기별 층수_숫자 분포 요약 ===
기준_년분기  min  max  p50  p99  지상70층초과_건수  지하10층초과_건수   유효값_비율
2022Q1 -7.0 86.0  1.0 14.0           6           0 0.639676
2022Q2 -7.0 82.0  1.0 14.0           5           0 0.642111
2022Q3 -8.0 82.0  1.0 15.0           5           0 0.644111
2022Q4 -8.0 86.0  1.0 15.0           6           0 0.645506
2023Q1 -9.0 86.0  1.0 15.0           6           0 0.673622
2023Q2 -9.0 86.0  1.0 15.0           7           0 0.676142
2023Q3 -9.0 86.0  1.0 15.0           7           0 0.676132
2023Q4 -9.0 86.0  1.0 15.0           5           0 0.678879
2024Q1 -9.0 86.0  1.0 15.0           5           0 0.679071
2024Q2 -9.0 86.0  1.0 15.0           5           0 0.678856
2024Q3 -9.0 86.0  1.0 15.0           5           0 0.677301
2024Q4 -8.0 86.0  1.0 15.0           5           0 0.667403
2025Q1 -8.0 86.0  1.0 15.0           5           0 0.670247
2025Q2 -8.0 86.0  1.0 15.0           4           0 0.670857
2025Q3 -8.0 86.0  1.0 15.0           4           0 0.650185
2025Q4 -8.0 86.0

# D. 피처 생성

In [10]:


from pathlib import Path

import numpy as np
import pandas as pd





def log(msg: str = "") -> None:
    print(msg)



ID_COL = "상가업소번호"
BUILDING_COL = "건물관리번호"
CATEGORY_COL = "상권업종소분류코드"

RADIUS_METERS_LIST = [100, 250]
EARTH_RADIUS_METERS = 6_371_000
MIN_SAMPLE_FOR_RELIABLE_DIVERSITY = 3  


def shannon_entropy(series: pd.Series) -> float:
    probs = series.value_counts(normalize=True).values
    probs = probs[probs > 0]
    return float(-(probs * np.log(probs)).sum())


def build_building_features(df: pd.DataFrame) -> pd.DataFrame:
    valid = df.dropna(subset=[BUILDING_COL])

    count_map = valid.groupby(BUILDING_COL)[ID_COL].count()
    nunique_map = valid.groupby(BUILDING_COL)[CATEGORY_COL].nunique()
    entropy_map = valid.groupby(BUILDING_COL)[CATEGORY_COL].apply(shannon_entropy)

    building_total = df[BUILDING_COL].map(count_map)
    building_diversity_unique = df[BUILDING_COL].map(nunique_map)
    building_diversity_entropy = df[BUILDING_COL].map(entropy_map)

    return pd.DataFrame({
        "건물내총업소수": building_total,
        "건물내업종다양성_unique": building_diversity_unique,
        "건물내업종다양성_entropy": building_diversity_entropy,
        "건물내표본수부족여부": (building_total < MIN_SAMPLE_FOR_RELIABLE_DIVERSITY).astype("Int64"),
        "건물관리번호결측여부": df[BUILDING_COL].isna().astype(int),
    }, index=df.index)


def build_radius_diversity_features(df: pd.DataFrame) -> pd.DataFrame:
    try:
        from sklearn.neighbors import BallTree
    except ImportError as e:
        raise ImportError(
            "scikit-learn이 필요합니다. 'pip install scikit-learn' 실행 후 다시 시도하세요."
        ) from e

    result_cols = {}
    for r in RADIUS_METERS_LIST:
        result_cols[f"반경{r}m_업종다양성"] = np.full(len(df), np.nan)
        result_cols[f"반경{r}m_이웃수"] = np.full(len(df), np.nan)

    valid_mask = df["위도"].notna() & df["경도"].notna()
    valid_df = df.loc[valid_mask]

    if valid_df.empty:
        return pd.DataFrame(result_cols, index=df.index)

    coords_rad = np.radians(valid_df[["위도", "경도"]].values)
    tree = BallTree(coords_rad, metric="haversine")

    categories = valid_df[CATEGORY_COL].astype(str)
    cat_to_idx = {c: i for i, c in enumerate(categories.unique())}
    cat_idx_arr = categories.map(cat_to_idx).values

    positional_to_original_index = valid_df.index.to_numpy()

    for r in RADIUS_METERS_LIST:
        radius_rad = r / EARTH_RADIUS_METERS
        neighbor_lists = tree.query_radius(coords_rad, r=radius_rad)

        entropies = np.full(len(valid_df), np.nan)
        neighbor_counts = np.full(len(valid_df), np.nan)

        for i, neighbors in enumerate(neighbor_lists):
            neighbors = neighbors[neighbors != i]  
            neighbor_counts[i] = len(neighbors)
            if len(neighbors) == 0:
                entropies[i] = np.nan  
                continue
            cats = cat_idx_arr[neighbors]
            counts = np.bincount(cats)
            probs = counts[counts > 0] / counts.sum()
            entropies[i] = float(-np.sum(probs * np.log(probs)))

        result_cols[f"반경{r}m_업종다양성"][positional_to_original_index] = entropies
        result_cols[f"반경{r}m_이웃수"][positional_to_original_index] = neighbor_counts

    return pd.DataFrame(result_cols, index=df.index)


def build_features_for_quarter(key: str) -> pd.DataFrame:
    path = CLEAN_DIR / f"{key}.pkl"
    df = pd.read_pickle(path)

    building_feats = build_building_features(df)
    radius_feats = build_radius_diversity_features(df)

    result = pd.concat([df[[ID_COL]], building_feats, radius_feats], axis=1)
    result.insert(1, "기준_년분기", key)

    log(f"[{key}] 완료 ({len(result):,}행)")
    log(f"    건물내총업소수 평균: {result['건물내총업소수'].mean():.2f}, "
        f"건물관리번호결측 비율: {result['건물관리번호결측여부'].mean():.2%}")
    log(f"    건물내표본수부족(<{MIN_SAMPLE_FOR_RELIABLE_DIVERSITY}) 비율: "
        f"{result['건물내표본수부족여부'].mean():.2%}")
    for r in RADIUS_METERS_LIST:
        col = f"반경{r}m_업종다양성"
        cnt_col = f"반경{r}m_이웃수"
        log(f"    {col} 평균: {result[col].mean():.3f} (결측 {result[col].isna().mean():.2%}), "
            f"평균 이웃수: {result[cnt_col].mean():.1f}")

    del df
    return result


def run_step_d() -> None:
    keys = sorted(p.stem for p in CLEAN_DIR.glob("*.pkl"))

    log(f"[분기 목록] {keys}\n")

    for key in keys:
        out_path = FEATURES_DIR / f"{key}.pkl"

        if out_path.exists():
            log(f"[{key}] 기존 결과 존재 → 계산 건너뜀")
            log(f"    기존 파일: {out_path}\n")
            continue

        feats = build_features_for_quarter(key)
        feats.to_pickle(out_path)

        log(f"    저장 완료: {out_path}\n")
        del feats



if __name__ == "__main__":
    run_step_d()

[분기 목록] ['2022Q1', '2022Q2', '2022Q3', '2022Q4', '2023Q1', '2023Q2', '2023Q3', '2023Q4', '2024Q1', '2024Q2', '2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4']

[2022Q1] 완료 (524,728행)
    건물내총업소수 평균: 34.15, 건물관리번호결측 비율: 0.66%
    건물내표본수부족(<3) 비율: 25.61%
    반경100m_업종다양성 평균: 3.414 (결측 0.13%), 평균 이웃수: 142.5
    반경250m_업종다양성 평균: 4.033 (결측 0.01%), 평균 이웃수: 598.2
    저장 완료: .\data\features_d\2022Q1.pkl

[2022Q2] 완료 (529,441행)
    건물내총업소수 평균: 34.39, 건물관리번호결측 비율: 0.64%
    건물내표본수부족(<3) 비율: 25.48%
    반경100m_업종다양성 평균: 3.417 (결측 0.13%), 평균 이웃수: 143.7
    반경250m_업종다양성 평균: 4.035 (결측 0.01%), 평균 이웃수: 603.6
    저장 완료: .\data\features_d\2022Q2.pkl

[2022Q3] 완료 (534,473행)
    건물내총업소수 평균: 34.90, 건물관리번호결측 비율: 0.62%
    건물내표본수부족(<3) 비율: 25.25%
    반경100m_업종다양성 평균: 3.421 (결측 0.13%), 평균 이웃수: 145.4
    반경250m_업종다양성 평균: 4.037 (결측 0.01%), 평균 이웃수: 610.3
    저장 완료: .\data\features_d\2022Q3.pkl

[2022Q4] 완료 (534,294행)
    건물내총업소수 평균: 34.89, 건물관리번호결측 비율: 0.59%
    건물내표본수부족(<3) 비율: 25.24%
    반경100m_업종다양성 

# E. 결합

In [11]:
from pathlib import Path

import numpy as np
import pandas as pd
import re



def log(msg: str = "") -> None:
    print(msg)




ID_COL = "상가업소번호"
QUARTER_COL = "기준_년분기"

COLUMN_DESCRIPTIONS = {
    "폐업_다음분기": "타깃. 1=다음 분기에 폐업 확정, 0=생존/이전/데이터갭",
    "라벨_비고": "폐업/생존/이전추정/데이터갭보정 중 하나 (B단계 라벨링 근거)",
    "이상치_의심분기": "IQR 기준 통계적 이상치로 탐지된 분기 여부 (2022Q4 등). 학습 시 포함/제외/가중치 조정 검토 필요",
    "지점여부": "지점명 존재 여부 (주의: 프랜차이즈 여부와 동일하지 않음)",
    "건물명존재여부": "건물명 등록 여부 (주의: 건물 규모/용도를 직접 의미하지 않음)",
    "층수_숫자": "파싱된 층수(지상 양수/지하 음수). '층+호' 압축 표기는 층만 추정 분리함",
    "지상1층여부": "층수_숫자==1 인 경우 1",
    "지하여부": "층수_숫자<0 이거나 지하 텍스트가 명시된 경우 1",
    "반지하여부": "반지하 표기('반', '반지층') 인 경우 1",
    "지하층수미상여부": "지하는 맞으나 정확한 깊이를 알 수 없는 경우('지','지층') 1",
    "층호결합추정여부": "3~4자리 숫자를 '층+호' 압축 표기로 판단해 층수를 추정 분리한 경우 1",
    "층정보파싱실패여부": "위 규칙으로도 해석 불가능한 층정보 표기(거의 0%)",
    "층정보결측여부": "층정보 원본이 결측인 경우 1",
    "건물내총업소수": "같은 건물관리번호 내 전체 업소 수 (해당 분기 기준)",
    "건물내업종다양성_unique": "같은 건물 내 업종소분류 종류 수",
    "건물내업종다양성_entropy": "같은 건물 내 업종 분포의 섀넌 엔트로피",
    "건물내표본수부족여부": "건물 내 업소 수가 3개 미만이라 다양성 지표 신뢰도가 낮은 경우 1",
    "건물관리번호결측여부": "건물관리번호 자체가 없어 건물 관련 피처가 NaN인 경우 1",
    "반경100m_업종다양성": "좌표 기준 반경 100m 이웃 업소들의 업종 엔트로피 (경쟁강도와 다른 개념)",
    "반경100m_이웃수": "반경 100m 내 실제 이웃 수 (엔트로피값 신뢰도 판단용)",
    "반경250m_업종다양성": "좌표 기준 반경 250m 이웃 업소들의 업종 엔트로피",
    "반경250m_이웃수": "반경 250m 내 실제 이웃 수",
}


def resolve_path(base_dir: Path, key: str) -> Path:
    for suffix in (".pkl", ".parquet"):
        p = base_dir / f"{key}{suffix}"
        if p.exists():
            return p
    raise FileNotFoundError(f"{key} 파일을 {base_dir}에서 찾을 수 없습니다.")


def load(base_dir: Path, key: str) -> pd.DataFrame:
    path = resolve_path(base_dir, key)
    if path.suffix == ".pkl":
        return pd.read_pickle(path)
    return pd.read_parquet(path)

def build_quarter_feature_table(key: str) -> pd.DataFrame:
    clean_df = load(CLEAN_DIR, key).reset_index(drop=True)
    feat_df = load(FEATURES_DIR, key).reset_index(drop=True)

    if len(clean_df) != len(feat_df):
        raise ValueError(
            f"[{key}] 행 개수 불일치: "
            f"clean={len(clean_df):,}, feature={len(feat_df):,}"
        )

    clean_ids = clean_df[ID_COL].astype("string")
    feat_ids = feat_df[ID_COL].astype("string")

    if not clean_ids.equals(feat_ids):
        raise ValueError(f"[{key}] 행별 상가업소번호 순서가 다릅니다.")

    overlapping_cols = [
        col for col in feat_df.columns
        if col in clean_df.columns
    ]

    feat_cols_only = feat_df.drop(columns=overlapping_cols)

    merged = pd.concat(
        [clean_df, feat_cols_only],
        axis=1,
    )

    return merged

def run_final_merge() -> None:
    train_path = FINAL_DIR / "train_table.pkl"
    predict_path = FINAL_DIR / "predict_target.pkl"

    if train_path.exists() and predict_path.exists():
        log("[체크포인트 발견] 기존 최종 결과를 불러옵니다.")

        train_table = pd.read_pickle(train_path)
        predict_target = pd.read_pickle(predict_path)

        log(f"[학습용 테이블 로드] {train_table.shape}")
        log(f"[예측 대상 테이블 로드] {predict_target.shape}")
        log(f"[기존 파일] {train_path}")
        log(f"[기존 파일] {predict_path}")

        return train_table, predict_target

    keys = sorted(
    p.stem
    for p in CLEAN_DIR.glob("*.pkl")
    if re.fullmatch(r"20\d{2}Q[1-4]", p.stem)
)
    log(f"[분기 목록] {keys}\n")

    all_quarters = []

    for key in keys:
        merged = build_quarter_feature_table(key)
        log(f"[{key}] C+D 병합 완료: {merged.shape}")

        all_quarters.append(merged)
        del merged

    features_all = pd.concat(all_quarters, ignore_index=True)
    del all_quarters

    log(f"\n[전체 피처 결합] {features_all.shape}")

    labels = pd.read_pickle(
        LABELS_DIR / "all_labels_combined.pkl"
    )
    log(f"[라벨 로드] {labels.shape}")

    features_all["_중복순번"] = (
    features_all
    .groupby([ID_COL, QUARTER_COL])
    .cumcount()
)

    labels["_중복순번"] = (
        labels
        .groupby([ID_COL, QUARTER_COL])
        .cumcount()
)

    train_table = features_all.merge(
        labels,
        on=[ID_COL, QUARTER_COL, "_중복순번"],
        how="inner",
        validate="one_to_one",
)

    train_table = train_table.drop(columns=["_중복순번"])
    features_all = features_all.drop(columns=["_중복순번"])


    
    log(f"\n[학습용 테이블] {train_table.shape}")

    last_key = keys[-1]

    predict_target = features_all[
        features_all[QUARTER_COL] == last_key
    ].copy()

    log(
        f"[예측 대상 테이블] "
        f"({last_key}) {predict_target.shape}"
    )

    train_table.to_pickle(train_path)
    predict_target.to_pickle(predict_path)

    log(f"\n[저장 완료] {train_path}")
    log(f"[저장 완료] {predict_path}")

    sample_n = min(2000, len(train_table))

    sample = train_table.sample(
        n=sample_n,
        random_state=42,
    )

    sample_path = FINAL_DIR / "train_table_sample.csv"

    sample.to_csv(
        sample_path,
        index=False,
        encoding="utf-8-sig",
    )

    log(
        f"[저장 완료] {sample_path} "
        f"({sample_n:,}행 랜덤 샘플)"
    )

    dict_rows = []

    for i, col in enumerate(train_table.columns):
        series = train_table.iloc[:, i]

        dict_rows.append({
            "컬럼명": col,
            "dtype": str(series.dtype),
            "결측률": series.isna().mean(),
            "설명": COLUMN_DESCRIPTIONS.get(
                col,
                "원본 소진공 상가업소 데이터 컬럼 그대로",
        ),  
    })

    data_dict = pd.DataFrame(dict_rows)

    dict_path = FINAL_DIR / "data_dictionary.csv"

    data_dict.to_csv(
        dict_path,
        index=False,
        encoding="utf-8-sig",
    )

    log(f"[저장 완료] {dict_path}")

    log("\n=== 최종 요약 ===")
    log(
        f"학습용 테이블: "
        f"{train_table.shape[0]:,}행 "
        f"x {train_table.shape[1]}컬럼"
    )
    log(
        f"폐업(1) 비율: "
        f"{train_table['폐업_다음분기'].mean():.4%}"
    )
    log(
        f"예측 대상 테이블({last_key}): "
        f"{predict_target.shape[0]:,}행"
    )


    return train_table, predict_target


if __name__ == "__main__":
    train_table, predict_target = run_final_merge()

[분기 목록] ['2022Q1', '2022Q2', '2022Q3', '2022Q4', '2023Q1', '2023Q2', '2023Q3', '2023Q4', '2024Q1', '2024Q2', '2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4']

[2022Q1] C+D 병합 완료: (524728, 54)
[2022Q2] C+D 병합 완료: (529441, 54)
[2022Q3] C+D 병합 완료: (534473, 54)
[2022Q4] C+D 병합 완료: (534294, 54)
[2023Q1] C+D 병합 완료: (474260, 54)
[2023Q2] C+D 병합 완료: (489903, 54)
[2023Q3] C+D 병합 완료: (482959, 54)
[2023Q4] C+D 병합 완료: (489872, 54)
[2024Q1] C+D 병합 완료: (484226, 54)
[2024Q2] C+D 병합 완료: (477378, 54)
[2024Q3] C+D 병합 완료: (466022, 54)
[2024Q4] C+D 병합 완료: (553535, 54)
[2025Q1] C+D 병합 완료: (540516, 54)
[2025Q2] C+D 병합 완료: (536114, 54)
[2025Q3] C+D 병합 완료: (536261, 54)
[2025Q4] C+D 병합 완료: (534978, 54)

[전체 피처 결합] (8188960, 54)
[라벨 로드] (7653982, 5)

[학습용 테이블] (7653982, 57)
[예측 대상 테이블] (2025Q4) (534978, 54)

[저장 완료] .\data\final\train_table.pkl
[저장 완료] .\data\final\predict_target.pkl
[저장 완료] .\data\final\train_table_sample.csv (2,000행 랜덤 샘플)
[저장 완료] .\data\final\data_dictionary.csv

=== 최종 요약 ===
학습용

# 인허가 데이터 전처리 (8개 업종 병합 및 드랍)

# 2. 8개 병합 + 각 결측치 확인

In [12]:
import pandas as pd
import os

try:
    import chardet
    HAS_CHARDET = True
except ImportError:
    HAS_CHARDET = False


UPLOAD_DIR = BASE_DIR
OUTPUT_DIR = BASE_DIR

FILES = {
    "일반음식점": "서울시 일반음식점 인허가 정보.csv",
    "휴게음식점": "서울시 휴게음식점 인허가 정보.csv",
    "이용업": "서울시 이용업 인허가 정보.csv",
    "미용업": "서울시 미용업 인허가 정보.csv",
    "담배소매업": "서울시 담배소매업 인허가 정보.csv",
    "의원": "서울시 의원 인허가 정보.csv",
    "종합체육시설업": "서울시 종합체육시설업 인허가 정보.csv",
    "통신판매업": "서울시 통신판매업 인허가 정보.csv",
}

ENCODING_CANDIDATES = ["cp949", "euc-kr", "utf-8"]


def detect_encoding(path: str) -> str:
    if not HAS_CHARDET:
        return None
    with open(path, "rb") as f:
        raw = f.read(200_000)  
    result = chardet.detect(raw)
    return result.get("encoding")


def load_csv_safely(path: str) -> pd.DataFrame:
    errors_log = []

    candidates = list(ENCODING_CANDIDATES)
    detected = detect_encoding(path)
    if detected and detected.lower() not in [c.lower() for c in candidates]:
        candidates = [detected] + candidates

    for enc in candidates:
        try:
            df = pd.read_csv(path, encoding=enc, low_memory=False)
            print(f"    -> 인코딩 '{enc}'로 로드 성공")
            return df
        except Exception as e:
            errors_log.append(f"    - '{enc}' 시도 실패: {type(e).__name__}: {e}")

    print(f"[경고] {path} : 모든 인코딩 정상 디코딩 실패. 시도 내역:")
    for line in errors_log:
        print(line)
    print(f"    -> cp949 + errors='replace'로 강제 로드 (디코딩 안 되는 극소수 문자만 깨질 수 있음)")
    return pd.read_csv(path, encoding="cp949", encoding_errors="replace", low_memory=False)


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True) 

    raw_dfs = []          
    missing_reports = []  

    for industry, filename in FILES.items():
        path = os.path.join(UPLOAD_DIR, filename)
        if not os.path.exists(path):
            print(f"[스킵] {industry}: 파일 없음 -> {path}")
            continue

        df = load_csv_safely(path)

        n_rows = len(df)
        miss_count = df.isna().sum()
        miss_ratio = (miss_count / n_rows * 100).round(2)

        report = pd.DataFrame({
            "업종": industry,
            "컬럼": miss_count.index,
            "결측수": miss_count.values,
            "전체행수": n_rows,
            "결측률(%)": miss_ratio.values,
        })
        missing_reports.append(report)

        df = df.copy()
        df["업종_대분류"] = industry
        raw_dfs.append(df)

        print(f"[완료] {industry}: {n_rows}행, {df.shape[1]}컬럼 로드")

    if not raw_dfs:
        print("로드된 파일이 없습니다. FILES의 파일명과 실제 업로드 파일명을 확인해주세요.")
        return

    missing_summary = pd.concat(missing_reports, ignore_index=True)
    missing_summary_path = os.path.join(OUTPUT_DIR, "missing_summary_by_industry.csv")
    missing_summary.to_csv(missing_summary_path, index=False, encoding="utf-8-sig")
    print(f"\n[저장] 업종별 결측 현황 -> {missing_summary_path}")

    merged_raw = pd.concat(raw_dfs, ignore_index=True, sort=False)
    merged_raw_path = os.path.join(OUTPUT_DIR, "merged_raw_8industries.csv")
    merged_raw.to_csv(merged_raw_path, index=False, encoding="utf-8-sig")
    print(f"[저장] raw 병합 데이터셋 -> {merged_raw_path}")
    print(f"  전체 행수: {len(merged_raw)}, 전체 컬럼수: {merged_raw.shape[1]}")

    col_presence = {}
    for df in raw_dfs:
        industry = df["업종_대분류"].iloc[0]
        for c in df.columns:
            if c == "업종_대분류":
                continue
            col_presence.setdefault(c, set()).add(industry)

    presence_df = pd.DataFrame([
        {"컬럼": c, "존재하는_업종수": len(v), "존재하는_업종": ", ".join(sorted(v))}
        for c, v in col_presence.items()
    ]).sort_values("존재하는_업종수")
    presence_path = os.path.join(OUTPUT_DIR, "column_presence_by_industry.csv")
    presence_df.to_csv(presence_path, index=False, encoding="utf-8-sig")
    print(f"[저장] 컬럼별 업종 존재 현황(구조적 결측 판단용) -> {presence_path}")


if __name__ == "__main__":
    main()

[경고] .\data\서울시 일반음식점 인허가 정보.csv : 모든 인코딩 정상 디코딩 실패. 시도 내역:
    - 'cp949' 시도 실패: UnicodeDecodeError: 'cp949' codec can't decode byte 0x82 in position 84091: illegal multibyte sequence
    - 'euc-kr' 시도 실패: UnicodeDecodeError: 'euc_kr' codec can't decode byte 0x9f in position 177696: illegal multibyte sequence
    - 'utf-8' 시도 실패: UnicodeDecodeError: 'utf-8' codec can't decode byte 0xb0 in position 0: invalid start byte
    -> cp949 + errors='replace'로 강제 로드 (디코딩 안 되는 극소수 문자만 깨질 수 있음)
[완료] 일반음식점: 535603행, 40컬럼 로드
[경고] .\data\서울시 휴게음식점 인허가 정보.csv : 모든 인코딩 정상 디코딩 실패. 시도 내역:
    - 'cp949' 시도 실패: UnicodeDecodeError: 'cp949' codec can't decode byte 0x82 in position 76708: illegal multibyte sequence
    - 'euc-kr' 시도 실패: UnicodeDecodeError: 'euc_kr' codec can't decode byte 0xa0 in position 218225: illegal multibyte sequence
    - 'utf-8' 시도 실패: UnicodeDecodeError: 'utf-8' codec can't decode byte 0xb0 in position 0: invalid start byte
    -> cp949 + errors='replace'로 강제 로드 (디코딩 안 되는 극소수 문자만 깨질

In [13]:
import pandas as pd
import os

PATH = BASE_DIR /"서울시 담배소매업 인허가 정보.csv"

df = pd.read_csv(PATH, encoding="cp949", low_memory=False)

col_a = "인허가일자"
col_b = "지정일자"

n_total = len(df)
n_na_a = df[col_a].isna().sum()
n_na_b = df[col_b].isna().sum()

exact_match = (df[col_a] == df[col_b])
n_match = exact_match.sum()
n_mismatch = n_total - n_match

print(f"전체 행수: {n_total}")
print(f"'{col_a}' 결측: {n_na_a}건")
print(f"'{col_b}' 결측: {n_na_b}건")
print(f"두 값이 정확히 일치하는 행: {n_match}건 ({n_match/n_total*100:.2f}%)")
print(f"두 값이 다른(또는 한쪽만 결측인) 행: {n_mismatch}건 ({n_mismatch/n_total*100:.2f}%)")

df["_a_dt"] = pd.to_datetime(df[col_a], errors="coerce")
df["_b_dt"] = pd.to_datetime(df[col_b], errors="coerce")
date_match = (df["_a_dt"] == df["_b_dt"])
n_date_match = date_match.sum()
print(f"\n[날짜형 변환 후 비교] 일치: {n_date_match}건 ({n_date_match/n_total*100:.2f}%)")

mismatch_rows = df.loc[~exact_match, [col_a, col_b]].head(20)
print("\n[불일치 샘플 20건]")
print(mismatch_rows)

OUTPUT_DIR = BASE_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)
mismatch_all = df.loc[~exact_match, [col_a, col_b]]
mismatch_all.to_csv(os.path.join(OUTPUT_DIR, "지정일자_인허가일자_불일치.csv"),
                     index=False, encoding="utf-8-sig")
print(f"\n[저장] 불일치 전체 목록 -> {OUTPUT_DIR}/지정일자_인허가일자_불일치.csv")

전체 행수: 95329
'인허가일자' 결측: 0건
'지정일자' 결측: 0건
두 값이 정확히 일치하는 행: 72387건 (75.93%)
두 값이 다른(또는 한쪽만 결측인) 행: 22942건 (24.07%)

[날짜형 변환 후 비교] 일치: 72387건 (75.93%)

[불일치 샘플 20건]
                   인허가일자                  지정일자
4   2002-05-10                                
5   2002-05-01                                
6   2003-03-03                                
11  2001-08-29                                
14  2004-02-24                                
15  2002-04-19                                
17  2000-10-02                                
21  1998-12-26                                
32  2000-02-28                                
33  2004-03-04                                
34  1999-10-01                                
37  2004-10-13                                
40  1975-07-24                                
41  1998-01-01                                
42  2003-07-14                                
43  2003-08-22                                
47  2000-01-11                        

# 3. 드랍

In [ ]:
import pandas as pd
import os

OUTPUT_DIR = BASE_DIR
RAW_PATH = os.path.join(OUTPUT_DIR, "merged_raw_8industries.csv")

MISSING_REMOVED_PATH = os.path.join(OUTPUT_DIR, "missing_removed_8industries.csv")
FINAL_PATH = os.path.join(OUTPUT_DIR, "final_8industries.csv")

DROP_COLUMNS = [
    "좌석수",
    "홈페이지",
    "전통업소지정번호",
    "전통업소주된음식",
    "본사종업원수",
    "공장사무직종업원수",
    "공장판매직종업원수",
    "공장생산직종업원수",
    "조건부허가신고사유",
    "조건부허가시작일자",
    "조건부허가종료일자",
    "보증액",
    "월세액",
    "민원종류명",
    "지정일자",
]


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    if os.path.exists(MISSING_REMOVED_PATH) and os.path.exists(FINAL_PATH):
        print("[체크포인트 발견] 기존 결과 파일을 사용합니다.")
        print(f"  결측행 제거본: {MISSING_REMOVED_PATH}")
        print(f"  최종 데이터셋: {FINAL_PATH}")

        df_missing_removed = pd.read_csv(
            MISSING_REMOVED_PATH,
            encoding="utf-8-sig",
            low_memory=False,
        )

        df_final = pd.read_csv(
            FINAL_PATH,
            encoding="utf-8-sig",
            low_memory=False,
        )

        print("\n===== 기존 결과 요약 =====")
        print(
            f"결측행 제거본: "
            f"{len(df_missing_removed)}행 x "
            f"{df_missing_removed.shape[1]}컬럼"
        )
        print(
            f"최종 데이터셋: "
            f"{len(df_final)}행 x "
            f"{df_final.shape[1]}컬럼"
        )

        return df_missing_removed, df_final

    print(f"[로드] {RAW_PATH}")

    df = pd.read_csv(
        RAW_PATH,
        encoding="utf-8-sig",
        low_memory=False,
    )

    n_before = len(df)

    print(
        f"  원본 행수: {n_before}, "
        f"컬럼수: {df.shape[1]}"
    )

    mask_no_name = df["사업장명"].isna()
    mask_no_address = (
        df["지번주소"].isna()
        & df["도로명주소"].isna()
    )

    n_no_name = mask_no_name.sum()
    n_no_address = mask_no_address.sum()
    n_drop_rows = (
        mask_no_name | mask_no_address
    ).sum()

    print("\n[결측행 확인]")
    print(f"  사업장명 결측: {n_no_name}건")
    print(
        f"  지번주소+도로명주소 둘 다 결측: "
        f"{n_no_address}건"
    )
    print(
        f"  제거 대상 행(중복 제외): "
        f"{n_drop_rows}건"
    )

    df_missing_removed = df.loc[
        ~(mask_no_name | mask_no_address)
    ].copy()

    n_after = len(df_missing_removed)

    print(
        f"  제거 후 행수: {n_after} "
        f"(제거된 행: {n_before - n_after}건)"
    )

    df_missing_removed.to_csv(
        MISSING_REMOVED_PATH,
        index=False,
        encoding="utf-8-sig",
    )

    print(
        f"[저장] 결측행 제거 데이터셋 "
        f"-> {MISSING_REMOVED_PATH}"
    )

    existing_drop_cols = [
        c
        for c in DROP_COLUMNS
        if c in df_missing_removed.columns
    ]

    missing_drop_cols = [
        c
        for c in DROP_COLUMNS
        if c not in df_missing_removed.columns
    ]

    if missing_drop_cols:
        print(
            "\n[경고] 드랍 목록에 있지만 "
            f"데이터셋에 없는 컬럼: {missing_drop_cols}"
        )

    df_final = df_missing_removed.drop(
        columns=existing_drop_cols
    )

    print(
        f"\n[열 드랍] 제거된 컬럼 수: "
        f"{len(existing_drop_cols)}"
    )
    print(
        f"  드랍된 컬럼: "
        f"{existing_drop_cols}"
    )
    print(
        f"  최종 컬럼수: {df_final.shape[1]} "
        f"(기존 {df_missing_removed.shape[1]}개에서 감소)"
    )

    df_final.to_csv(
        FINAL_PATH,
        index=False,
        encoding="utf-8-sig",
    )

    print(
        f"[저장] 최종 데이터셋 "
        f"-> {FINAL_PATH}"
    )

    print("\n===== 요약 =====")
    print(
        f"1) raw 병합본: "
        f"{n_before}행 x {df.shape[1]}컬럼"
    )
    print(
        f"2) 결측행 제거본: "
        f"{n_after}행 x "
        f"{df_missing_removed.shape[1]}컬럼"
    )
    print(
        f"3) 최종(열 드랍)본: "
        f"{len(df_final)}행 x "
        f"{df_final.shape[1]}컬럼"
    )

    return df_missing_removed, df_final

if __name__ == "__main__":
    df_missing_removed, df_final = main()

[체크포인트 발견] 기존 결과 파일을 사용합니다.
  결측행 제거본: .\data\missing_removed_8industries.csv
  최종 데이터셋: .\data\final_8industries.csv

===== 기존 결과 요약 =====
결측행 제거본: 1866336행 x 65컬럼
최종 데이터셋: 1866336행 x 49컬럼


In [15]:
import pandas as pd
import os

FINAL_PATH = BASE_DIR/"final_8industries.csv"

df = pd.read_csv(FINAL_PATH, encoding="utf-8-sig", low_memory=False)
print(f"[로드] {FINAL_PATH} -> {len(df)}행 x {df.shape[1]}컬럼\n")


def check_pair(wide_col: str, narrow_col: str):
    missing_cols = [
        col for col in [wide_col, narrow_col]
        if col not in df.columns
    ]

    if missing_cols:
        print(f"[건너뜀] 존재하지 않는 컬럼: {missing_cols}")
        print()
        return

    print(f"===== {wide_col} vs {narrow_col} =====")

    total = len(df)
    n_narrow_notna = df[narrow_col].notna().sum()

    print(f"전체 행수: {total}")
    print(
        f"'{narrow_col}' 값이 있는 행수: "
        f"{n_narrow_notna} ({n_narrow_notna / total * 100:.2f}%)"
    )

    overlap = df.loc[
        df[narrow_col].notna(),
        [wide_col, narrow_col]
    ].copy()

    n_overlap = len(overlap)
    n_wide_also_notna = overlap[wide_col].notna().sum()
    n_wide_missing = n_overlap - n_wide_also_notna

    print(
        f"겹치는 범위({n_overlap}건) 중 "
        f"'{wide_col}'도 값이 있는 행: {n_wide_also_notna}건"
    )

    if n_wide_missing > 0:
        print(
            f"[주의] '{wide_col}'이 없는데 "
            f"'{narrow_col}'만 있는 행: {n_wide_missing}건"
        )

    both_notna = overlap.dropna(subset=[wide_col, narrow_col])

    if len(both_notna) == 0:
        print("둘 다 값이 있는 행이 없습니다.")
        print()
        return

    a_numeric = pd.to_numeric(
        both_notna[wide_col],
        errors="coerce"
    )
    b_numeric = pd.to_numeric(
        both_notna[narrow_col],
        errors="coerce"
    )

    both_numeric_ok = (
        a_numeric.notna()
        & b_numeric.notna()
    )

    numeric_match = pd.Series(False, index=both_notna.index)
    string_match = pd.Series(False, index=both_notna.index)

    if both_numeric_ok.any():
        numeric_match.loc[both_numeric_ok] = (
            a_numeric.loc[both_numeric_ok]
            - b_numeric.loc[both_numeric_ok]
        ).abs() < 1e-6

    non_numeric_mask = ~both_numeric_ok

    if non_numeric_mask.any():
        string_match.loc[non_numeric_mask] = (
            both_notna.loc[non_numeric_mask, wide_col]
            .astype(str)
            .str.strip()
            ==
            both_notna.loc[non_numeric_mask, narrow_col]
            .astype(str)
            .str.strip()
        )

    match_mask = numeric_match | string_match
    n_match = int(match_mask.sum())
    n_mismatch = len(both_notna) - n_match

    print(
        f"둘 다 값이 있는 {len(both_notna)}건 중 "
        f"일치: {n_match}건 "
        f"({n_match / len(both_notna) * 100:.2f}%)"
    )
    print(f"불일치: {n_mismatch}건")
    print()

check_pair(wide_col="업태구분명", narrow_col="위생업태명")
check_pair(wide_col="소재지면적", narrow_col="시설총규모")

[로드] .\data\final_8industries.csv -> 1866336행 x 49컬럼

===== 업태구분명 vs 위생업태명 =====
전체 행수: 1866336
'위생업태명' 값이 있는 행수: 796906 (42.70%)
겹치는 범위(796906건) 중 '업태구분명'도 값이 있는 행: 796906건
둘 다 값이 있는 796906건 중 일치: 739912건 (92.85%)
불일치: 56994건

[건너뜀] 존재하지 않는 컬럼: ['시설총규모']



In [16]:
import pandas as pd
import os

FINAL_PATH = BASE_DIR/"final_8industries.csv"

EXTRA_DROP_COLUMN = "시설총규모"
EXTRA_DROP_REASON = "소재지면적과 완전 중복 + 결측을 0으로 채워넣어 왜곡 위험"

df = pd.read_csv(FINAL_PATH, encoding="utf-8-sig", low_memory=False)
n_before_cols = df.shape[1]
print(f"[로드] {FINAL_PATH} -> {len(df)}행 x {n_before_cols}컬럼")

if EXTRA_DROP_COLUMN in df.columns:
    df = df.drop(columns=[EXTRA_DROP_COLUMN])
    print(f"[드랍] '{EXTRA_DROP_COLUMN}' 제거 (사유: {EXTRA_DROP_REASON})")
else:
    print(f"[스킵] '{EXTRA_DROP_COLUMN}' 컬럼이 이미 없습니다.")

print(f"[결과] {len(df)}행 x {df.shape[1]}컬럼 (기존 {n_before_cols}컬럼에서 감소)")

df.to_csv(FINAL_PATH, index=False, encoding="utf-8-sig")
print(f"[저장] 덮어쓰기 완료 -> {FINAL_PATH}")

[로드] .\data\final_8industries.csv -> 1866336행 x 49컬럼
[스킵] '시설총규모' 컬럼이 이미 없습니다.
[결과] 1866336행 x 49컬럼 (기존 49컬럼에서 감소)
[저장] 덮어쓰기 완료 -> .\data\final_8industries.csv
